# Exercise 4 - Prompting
_By Abigail Hayes_

In this exercise you will work with [Hugging Face Transformers](https://huggingface.co/docs/transformers/index) to explore different approaches to prompting LLMs. We will focus on open-weight models that can be run locally (if you have a GPU available).

Note that the models we will use in this exercise are significantly larger (>1B parameters) than, e.g., distilbert (67M parameters). <br>
**This means that you will likely need to run this notebook on the BWUniCluster3.0 or on Google Colab to have enough GPU memory and compute.**

This exercise consists of three parts:
1. Prompt templates
2. CoT prompting
3. Persona prompting

## Setup

Follow the instructions for last week for setup. If you are using the BWUniCluster, just make sure that the same kernel as last week is selected for this notebook in the top right.

## 1. Prompt templates

We have already seen a prompt template last week. Here is another example where we set both the **system** prompt and a **user** prompt.

In [ ]:
from transformers import pipeline

# pipelines are a higher-level abstraction offered by the transformers library
# they can be used with both base models and instruction-tuned models
pipe = pipeline("text-generation", model="allenai/OLMo-2-0425-1B-Instruct")

messages = [
    {"role": "system", "content": "You are a rabbit."},
    {"role": "user", "content": "Describe yourself"}
]
pipe(messages)[0]['generated_text']

### System prompt

The **system** prompt instructs the model how to behave when provided with user prompts. They should define desirable behaviours (here 'being a rabbit') or try to constrain undesirable behaviours. Let's try again instructing the model to avoid a certain behaviour.

In [ ]:
messages = [
    {"role": "system", "content": "You are a rabbit. Do not discuss your fur."},
    {"role": "user", "content": "Describe yourself"}
]
pipe(messages)[0]['generated_text']

Now we will go further and ask the model to ignore physical characteristics completely.

In [ ]:
messages = [
    {"role": "system", "content": "You are a rabbit. Do not discuss your physical characteristics or features."},
    {"role": "user", "content": "Describe yourself"}
]
pipe(messages)[0]['generated_text']

Developing system prompts that avoid undesirable behaviour is not a simple process and you may need to try a number of variants. Let's try again!

In [ ]:
messages = [
    {"role": "system", "content": "You are a rabbit. Do not talk about your body, colour, body parts or fur."},
    {"role": "user", "content": "Describe yourself"}
]
pipe(messages)[0]['generated_text']

### User prompt

The **user** prompt is the specific question you want the model to answer. Once you have a system prompt, you will generally then provide many user prompts  for the same system prompt.

Let's try another user prompt option.

In [ ]:
messages = [
    {"role": "system", "content": "You are a rabbit. Do not talk about your body, colour, body parts or fur."},
    {"role": "user", "content": "What are you eating?"}
]
pipe(messages)[0]['generated_text']

As we saw with the system prompt, expressing the same concept in different ways can produce different results. So that we are confident with our final results, we often want to deliberately test the same question phrased in different ways. We can try this now.

In [ ]:
system_prompt = "You are a rabbit. Do not talk about your body, colour, body parts or fur."

messages = [
    {"role": "system", "content": system_prompt"},
    {"role": "user", "content": "What are you eating now?"}
]
pipe(messages)[0]['generated_text']

In [ ]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "What are you currently eating?"}
]
pipe(messages)[0]['generated_text']

## Task 1

Get the model to answer in English as a German actress. Use three different questions to ask about their favourite holiday destination.

## 2. CoT prompting

The exact type of prompting meant by Chain-of-Thought prompting can sometimes vary. The general idea is that the model should be prompted to answer after a series of reasoning steps, rather than just providing a stand alone answer.

We can start with the example from the lecture. First we prompt the model with the question and encourage it to only answer with a single number using both the system and user prompts.

In [ ]:
messages = [
    {"role": "system", "content": "You are a simple calculator."},
    {"role": "user", "content": "If a train travels 60 mph for 2 hours and 40 mph for 1 hour, how far does it travel in total? Answer with a single number."}
]
pipe(messages)[0]['generated_text']

Now we can try the same question but ask for step by step reasoning. We also tell the model it is a helpful AI assistant too.

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful AI assistant."},
    {"role": "user", "content": "If a train travels 60 mph for 2 hours and 40 mph for 1 hour, how far does it travel in total? Explain step by step."}
]
pipe(messages)[0]['generated_text']

There are other ways we might induce CoT reasoning, such as few-shot prompting.

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful AI assistant."},
    {"role": "user", "content": "If a train travels 75 mph for 1 hour and 80 mph for 3 hours, how far does it travel in total?"},
    {"role": "assistant", "content": " 75*1+80*3=75+240=315 miles"},
    {"role": "user", "content": "If a train travels 60 mph for 2 hours and 40 mph for 1 hour, how far does it travel in total?"}
]
pipe(messages)[0]['generated_text']

## Task 2

Try to get the model to use CoT reasoning to solve this task:

---

1. Every Humpus is a Grumpus.

2. Every Grumpus likes rain.

3. Some Glumps are Humpuses.

4. No Glump likes sun.

Decide whether each conclusion is Must be true, Could be true, or Cannot be true.

A. Every Humpus likes rain.

B. Some Glumps like rain.

C. No Humpus likes sun.

D. Some Grumpuses do not like sun.

---

The correct solution is: 

A. Must be true.

B. Must be true.

C. Could be true.

D. Must be true.

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful AI assistant."},
    {"role": "user", "content": "1. Every Humpus is a Grumpus. 2. Every Grumpus likes rain. 3. Some Glumps are Humpuses. 4. No Glump likes sun. Decide whether each conclusion is Must be true, Could be true, or Cannot be true. A. Every Humpus likes rain B. Some Glumps like rain. C. No Humpus likes sun. D. Some Grumpuses do not like sun."}
]
pipe(messages)[0]['generated_text']

## 3. Persona prompting

We have already seen basic persona prompting when instructing the model that it is a pirate or a rabbit. This information has clearly guided the model's tone and behaviour. Now let's explore this further.

First set up the prompts. We will look at a couple of different characteristic settings.

In [ ]:
import itertools
import pandas as pd

ages = [20, 80]
genders = ["man", "woman"]
countries = ["France", "Germany"]

system_prompts = []
characteristics = []  # to keep the parameters alongside the prompt
for age, gender, country in itertools.product(ages, genders, countries):
    sp = f"You are a {age} year old {gender} from {country}. Reply in English."
    system_prompts.append(sp)
    characteristics.append((age, gender, country))

user_prompt = "Which public figure was your hero as a child? Reply in a short sentence."

print(system_prompts[0])

Now we create the batch of inputs and receive the outputs from the model.

In [ ]:
# build a batch of message lists
batch_inputs = [
    [
        {"role": "system", "content": sp},
        {"role": "user", "content": user_prompt}
    ]
    for sp in system_prompts
]

# pass the batch directly to the pipeline
outputs = pipe(batch_inputs, max_new_tokens=100)

Finally we will organise the results into a table.

In [ ]:
rows = []
for (age, gender, country), conversation in zip(characteristics, outputs):
    reply = conversation[0]['generated_text'][-1]['content']

    rows.append({
        "Age": age,
        "Gender": gender,
        "Country": country,
        "Model Output": reply
    })
    
df = pd.DataFrame(rows)
df

Just from these few responses we can see some of the influence of the different characteristics. However, the variation is not just a result of the different personas.

Let's try running two of the personas 10 times.

In [ ]:
batch_inputs = [
    [
        {"role": "system", "content": "You are a 20 year old man from France. Reply in English."},
        {"role": "user", "content": "Which public figure was your hero as a child? Reply in a short sentence."}
    ]
    for _ in range(10)
]

outputs = pipe(batch_inputs, max_new_tokens=100)

rows = []
for conversation in outputs:
    reply = conversation[0]['generated_text'][-1]['content']

    print(reply)
    

In [ ]:
batch_inputs = [
    [
        {"role": "system", "content": "You are a 80 year old woman from Germany. Reply in English."},
        {"role": "user", "content": "Which public figure was your hero as a child? Reply in a short sentence."}
    ]
    for _ in range(10)
]

outputs = pipe(batch_inputs, max_new_tokens=100)

rows = []
for conversation in outputs:
    reply = conversation[0]['generated_text'][-1]['content']

    print(reply)

## Task 3

Construct personas by choosing 2 to 5 characteristics to vary over such that you would expect different results. 

Ask a model prompted with each of them the following 3 questions from the [World Value Survey (wave 7)](https://www.worldvaluessurvey.org/WVSDocumentationWV7.jsp):

Q119. Can you tell me how strongly you agree or disagree with the following statement: “on the whole, women are less corrupt than men”?

Q149. Most people consider both freedom and equality to be important, but if you had to choose between them, which one would you consider more important?

Q272. What language do you normally speak at home?

What patterns can you identify? You might need to do further text processing. Maybe you can plot a graph?